# Reranking with AWS OpenSearch & Bedrock

## 📖 Overview

This notebook demonstrates **Reranking** for RAG using AWS services:
- **AWS OpenSearch Serverless** for initial candidate retrieval
- **Amazon Bedrock Claude** for relevance-based reranking
- **Amazon Bedrock Titan** for embeddings

### What is Reranking?

Reranking improves retrieval quality through a two-stage process:
1. **Stage 1 (Fast Retrieval)**: Get top-N candidates using vector search (cheap, fast)
2. **Stage 2 (Reranking)**: Use LLM to score relevance of each candidate (expensive, accurate)
3. **Final Selection**: Select top-K from reranked results

### Why Rerank?

**Vector search limitations:**
- Relies on embedding similarity alone
- May miss nuanced relevance
- Can retrieve semantically similar but contextually irrelevant documents

**Reranking benefits:**
- LLM understands query intent deeply
- Considers query-document relevance explicitly
- Improves precision without sacrificing recall

### When to Use

✅ **Good for:**
- High-precision requirements
- Complex queries with subtle intent
- When top-K accuracy matters most
- Mitigating embedding model limitations
- Production RAG systems

❌ **Not ideal for:**
- Latency-critical applications (adds 1-3s)
- Very tight budgets (adds $0.02-0.05 per query)
- Simple factual lookups
- When vector search already performs well

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Generate Embedding<br/>Titan]
    B --> C[Stage 1: Fast Retrieval<br/>OpenSearch]
    C --> D[Get Top-20 Candidates<br/>Cast Wide Net]
    D --> E[Stage 2: LLM Reranking<br/>Claude Scores Each]
    E --> F[Sort by Relevance Score]
    F --> G[Select Top-5]
    G --> H[Generate Answer<br/>Claude + Context]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#ffe0b2
    style F fill:#fff9c4
    style G fill:#e8f5e9
    style H fill:#c8e6c9
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Tuple
import time
import numpy as np

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'reranking_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
RERANKER_MODEL = 'us.anthropic.claude-haiku-3-20241022-v1:0'  # Fast for scoring
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Quality for answers

# Reranking Parameters
RETRIEVAL_TOP_N = 20  # Stage 1: Cast wide net
FINAL_TOP_K = 5  # Stage 2: Narrow down

print(f"Configuration:")
print(f"  Stage 1 (Retrieval): Top-{RETRIEVAL_TOP_N}")
print(f"  Stage 2 (Reranking): Top-{FINAL_TOP_K}")
print(f"  Reranker: {RERANKER_MODEL.split('.')[-1][:20]}")
print(f"  Answer LLM: {LLM_MODEL.split('.')[-1][:20]}")

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
reranker = BedrockLLM(AWS_REGION, RERANKER_MODEL, temperature=0.1)  # Low temp for scoring
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

## 4️⃣ Load Sample Documents

We'll use documents where vector search might retrieve semantically similar but contextually less relevant results.

In [ ]:
sample_documents = [
    # Highly relevant to AWS RAG
    "AWS Bedrock provides managed access to foundation models including Claude Opus for complex reasoning and Claude Haiku for fast responses.",
    "Retrieval-Augmented Generation (RAG) with AWS combines OpenSearch Serverless vector search with Bedrock's Claude models for grounded answers.",
    "Amazon Titan Embeddings V2 generates 1024-dimensional vectors optimized for semantic search in OpenSearch.",
    
    # Moderately relevant
    "OpenSearch Serverless automatically scales compute and storage, eliminating infrastructure management for vector search workloads.",
    "Claude models use Constitutional AI training to produce helpful, harmless, and honest responses.",
    "Vector databases enable semantic search by storing embeddings and computing similarity scores.",
    
    # Lower relevance but semantically similar
    "Machine learning models can be deployed on AWS using SageMaker for custom training and inference.",
    "Natural language processing (NLP) enables computers to understand and generate human language.",
    "Cloud computing platforms like AWS provide scalable infrastructure for AI applications.",
    "Large language models are trained on massive text corpora to learn language patterns.",
    
    # Edge cases - similar terms but different context
    "Amazon Prime offers fast shipping and streaming services for subscribers.",
    "Bedrock geology studies the solid rock underlying soil and sediment layers.",
    "Claude Monet was a French impressionist painter known for his water lily series.",
    "Retrieval of lost data requires proper backup systems and recovery procedures.",
    "Vector graphics use mathematical formulas to represent images and scale without loss.",
    
    # More relevant technical docs
    "HNSW algorithm in OpenSearch enables fast approximate nearest neighbor search for vector embeddings.",
    "RAG systems improve answer accuracy by retrieving relevant context before generation.",
    "Bedrock API provides unified access to multiple foundation models with consistent authentication.",
    "Cosine similarity is the primary metric for comparing vector embeddings in semantic search.",
    "Claude Opus excels at multi-step reasoning tasks required for complex RAG queries."
]

print(f"Loaded {len(sample_documents)} documents")
print(f"\nDocument distribution:")
print(f"  Highly relevant: 3")
print(f"  Moderately relevant: 3")
print(f"  Lower relevance: 4")
print(f"  Edge cases: 5")
print(f"  Technical: 5")

## 5️⃣ Create Index and Index Documents

In [ ]:
# Create index
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

# Index documents
print("\nIndexing documents...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })
    if (i + 1) % 5 == 0:
        print(f"  Embedded {i + 1}/{len(sample_documents)}")

indexed = opensearch.index_documents(documents)
print(f"\n✓ Indexed {indexed} documents")

## 6️⃣ LLM-Based Reranker

Implement reranking using Claude to score query-document relevance.

### Scoring Approach

We ask Claude to score each document on a 0-10 scale:
- **0-3**: Not relevant
- **4-6**: Somewhat relevant
- **7-8**: Relevant
- **9-10**: Highly relevant

Claude evaluates:
- Topical relevance
- Query intent match
- Information usefulness
- Context appropriateness

In [ ]:
def rerank_documents(query: str, documents: List[Dict], top_k: int = 5) -> List[Dict]:
    """
    Rerank documents using LLM-based relevance scoring
    
    Args:
        query: User query
        documents: List of candidate documents
        top_k: Number of top documents to return
    
    Returns:
        Reranked documents with relevance scores
    """
    scored_docs = []
    
    print(f"Reranking {len(documents)} documents...")
    
    for i, doc in enumerate(documents, 1):
        # Create scoring prompt
        scoring_prompt = f"""
Rate the relevance of the following document to the query on a scale of 0-10.

Query: {query}

Document: {doc['text']}

Consider:
- Does the document directly answer or relate to the query?
- Is the information useful for addressing the query?
- Does it match the query's intent and context?

Respond with ONLY a number between 0-10, nothing else.
"""
        
        try:
            response = reranker.generate(scoring_prompt, temperature=0.1)
            # Extract numeric score
            score_str = response.strip().split()[0]  # Get first token
            score = float(score_str)
            score = max(0, min(10, score))  # Clamp to [0, 10]
        except:
            print(f"  Warning: Failed to score document {i}, using 0")
            score = 0.0
        
        doc_copy = doc.copy()
        doc_copy['rerank_score'] = score
        scored_docs.append(doc_copy)
        
        if i % 5 == 0:
            print(f"  Scored {i}/{len(documents)}")
    
    # Sort by rerank score
    scored_docs.sort(key=lambda x: x['rerank_score'], reverse=True)
    
    print(f"✓ Reranking complete\n")
    return scored_docs[:top_k]

# Test reranker with sample docs
test_query = "How does RAG work with AWS Bedrock?"
test_docs = [
    {'text': "AWS Bedrock provides managed access to foundation models for RAG applications.", 'id': '1'},
    {'text': "Bedrock geology involves the study of underground rock formations.", 'id': '2'},
    {'text': "RAG combines retrieval and generation for accurate responses.", 'id': '3'}
]

print("Testing reranker:\n")
reranked = rerank_documents(test_query, test_docs, top_k=3)

print("Reranked results:")
for i, doc in enumerate(reranked, 1):
    print(f"{i}. [Score: {doc['rerank_score']:.1f}] {doc['text'][:60]}...")

## 7️⃣ Complete Reranking Pipeline

Combine retrieval and reranking into a complete RAG system.

In [ ]:
def reranking_rag_query(query: str,
                        retrieval_top_n: int = 20,
                        final_top_k: int = 5) -> Tuple[str, List[Dict], List[Dict]]:
    """
    Query using two-stage retrieval with reranking
    
    Returns:
        (answer, reranked_docs, initial_docs)
    """
    start_time = time.time()
    
    # Stage 1: Fast vector retrieval
    print(f"Stage 1: Retrieving top-{retrieval_top_n} candidates...")
    query_embedding = embedder.embed_text(query)
    initial_results = opensearch.vector_search(query_embedding, top_k=retrieval_top_n)
    stage1_time = time.time() - start_time
    print(f"✓ Retrieved {len(initial_results)} candidates in {stage1_time:.2f}s\n")
    
    # Stage 2: LLM reranking
    print(f"Stage 2: Reranking to top-{final_top_k}...")
    rerank_start = time.time()
    reranked_results = rerank_documents(query, initial_results, top_k=final_top_k)
    stage2_time = time.time() - rerank_start
    print(f"Reranking took {stage2_time:.2f}s\n")
    
    # Stage 3: Generate answer
    print("Stage 3: Generating answer...")
    context_texts = [doc['text'] for doc in reranked_results]
    answer = llm.generate_with_context(query, context_texts)
    gen_time = time.time() - rerank_start - stage2_time
    
    total_time = time.time() - start_time
    print(f"✓ Answer generated in {gen_time:.2f}s\n")
    print(f"Total time: {total_time:.2f}s")
    
    return answer, reranked_results, initial_results

# Test reranking RAG
test_questions = [
    "How does RAG work with AWS Bedrock and Claude?",
    "What embedding models are available in AWS?",
    "Explain vector search in OpenSearch"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")
    print('='*70)
    
    answer, reranked, initial = reranking_rag_query(
        question,
        retrieval_top_n=RETRIEVAL_TOP_N,
        final_top_k=FINAL_TOP_K
    )
    
    print(f"\n📊 Retrieval Stats:")
    print(f"  Initial candidates: {len(initial)}")
    print(f"  After reranking: {len(reranked)}")
    
    print(f"\n📚 Top-{len(reranked)} Reranked Documents:")
    for j, doc in enumerate(reranked, 1):
        print(f"  {j}. [Rerank: {doc['rerank_score']:.1f}, Vector: {doc['score']:.3f}]")
        print(f"     {doc['text'][:70]}...")
    
    print(f"\n💡 Answer:")
    print(answer)

## 8️⃣ Comparison: Reranking vs Simple RAG

Compare results with and without reranking.

In [ ]:
comparison_query = "What is AWS Bedrock and how does it work with RAG?"

print("="*70)
print("WITH RERANKING")
print("="*70)

rerank_answer, reranked_docs, initial_docs = reranking_rag_query(
    comparison_query,
    retrieval_top_n=RETRIEVAL_TOP_N,
    final_top_k=FINAL_TOP_K
)

print("\n" + "="*70)
print("WITHOUT RERANKING (Simple RAG)")
print("="*70)

simple_embedding = embedder.embed_text(comparison_query)
simple_results = opensearch.vector_search(simple_embedding, top_k=FINAL_TOP_K)
simple_answer = llm.generate_with_context(comparison_query, [r['text'] for r in simple_results])

print(f"Retrieved {len(simple_results)} documents directly\n")

# Compare document IDs
reranked_ids = [doc['metadata']['doc_id'] for doc in reranked_docs]
simple_ids = [doc['metadata']['doc_id'] for doc in simple_results]

print("\n📊 COMPARISON:\n")
print("Top-5 Document IDs:")
print(f"  With Reranking:    {reranked_ids}")
print(f"  Without Reranking: {simple_ids}")

overlap = len(set(reranked_ids) & set(simple_ids))
print(f"\n  Overlap: {overlap}/{FINAL_TOP_K} documents")
print(f"  Different: {FINAL_TOP_K - overlap} documents")

print("\n🔍 Reranking Changes:")
for i in range(FINAL_TOP_K):
    if i < len(reranked_docs) and i < len(simple_results):
        rerank_id = reranked_docs[i]['metadata']['doc_id']
        simple_id = simple_results[i]['metadata']['doc_id']
        
        if rerank_id != simple_id:
            print(f"  Position {i+1}: Doc {simple_id} → Doc {rerank_id}")
            print(f"    Rerank score: {reranked_docs[i]['rerank_score']:.1f}")
            print(f"    Vector score: {reranked_docs[i]['score']:.3f}")

print("\n📝 Answer Quality:")
print(f"\nWith Reranking:\n{rerank_answer[:200]}...")
print(f"\nWithout Reranking:\n{simple_answer[:200]}...")

## 9️⃣ Reranking Impact Analysis

Analyze how reranking affects document ranking.

In [ ]:
import matplotlib.pyplot as plt

# Analyze a query in detail
analysis_query = "Explain RAG with AWS Bedrock"

print(f"Analyzing: {analysis_query}\n")

# Get results
query_emb = embedder.embed_text(analysis_query)
candidates = opensearch.vector_search(query_emb, top_k=15)
reranked = rerank_documents(analysis_query, candidates, top_k=15)

# Track position changes
print("Position Changes After Reranking:\n")
print(f"{'Rank':<6} {'Vector Score':<14} {'Rerank Score':<14} {'Change':<8} {'Doc Preview'}")
print("="*80)

for i, doc in enumerate(reranked, 1):
    # Find original position
    orig_pos = next((j for j, c in enumerate(candidates, 1) if c['id'] == doc['id']), None)
    change = orig_pos - i if orig_pos else 0
    change_str = f"+{change}" if change > 0 else str(change)
    
    print(f"{i:<6} {doc['score']:<14.4f} {doc['rerank_score']:<14.1f} {change_str:<8} {doc['text'][:40]}...")

# Visualize score comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Score comparison
ranks = list(range(1, len(reranked) + 1))
vector_scores = [doc['score'] for doc in reranked]
rerank_scores = [doc['rerank_score'] / 10 for doc in reranked]  # Normalize to 0-1

ax1.plot(ranks, vector_scores, 'o-', label='Vector Score', linewidth=2, markersize=8)
ax1.plot(ranks, rerank_scores, 's-', label='Rerank Score (scaled)', linewidth=2, markersize=8)
ax1.set_xlabel('Rank', fontsize=12)
ax1.set_ylabel('Score', fontsize=12)
ax1.set_title('Score Comparison: Vector vs Rerank', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Rank changes
position_changes = []
for i, doc in enumerate(reranked, 1):
    orig_pos = next((j for j, c in enumerate(candidates, 1) if c['id'] == doc['id']), i)
    change = orig_pos - i
    position_changes.append(change)

colors = ['green' if c > 0 else 'red' if c < 0 else 'gray' for c in position_changes]
ax2.barh(ranks, position_changes, color=colors, alpha=0.7)
ax2.set_xlabel('Position Change', fontsize=12)
ax2.set_ylabel('Final Rank', fontsize=12)
ax2.set_title('Rank Movement After Reranking', fontsize=14, fontweight='bold')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
ax2.grid(True, alpha=0.3, axis='x')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

print("\n📈 Key Metrics:")
print(f"  Average position change: {np.mean([abs(c) for c in position_changes]):.1f} positions")
print(f"  Documents moved up: {sum(1 for c in position_changes if c > 0)}")
print(f"  Documents moved down: {sum(1 for c in position_changes if c < 0)}")
print(f"  Documents unchanged: {sum(1 for c in position_changes if c == 0)}")

## 🔟 Performance & Cost Analysis

In [ ]:
# Detailed performance breakdown
test_query = "How does vector search work in OpenSearch?"

print("Performance & Cost Breakdown\n")
print("="*70)

# Time each stage
t1 = time.time()
query_emb = embedder.embed_text(test_query)
embed_time = time.time() - t1

t2 = time.time()
candidates = opensearch.vector_search(query_emb, top_k=RETRIEVAL_TOP_N)
retrieval_time = time.time() - t2

t3 = time.time()
reranked = rerank_documents(test_query, candidates, top_k=FINAL_TOP_K)
rerank_time = time.time() - t3

t4 = time.time()
answer = llm.generate_with_context(test_query, [r['text'] for r in reranked])
gen_time = time.time() - t4

total_time = embed_time + retrieval_time + rerank_time + gen_time

print("⏱️  Latency Breakdown:\n")
print(f"  1. Query Embedding:     {embed_time*1000:>8.1f}ms ({embed_time/total_time*100:>5.1f}%)")
print(f"  2. Vector Retrieval:    {retrieval_time*1000:>8.1f}ms ({retrieval_time/total_time*100:>5.1f}%)")
print(f"  3. Reranking ({RETRIEVAL_TOP_N} docs): {rerank_time*1000:>8.1f}ms ({rerank_time/total_time*100:>5.1f}%)")
print(f"  4. Answer Generation:   {gen_time*1000:>8.1f}ms ({gen_time/total_time*100:>5.1f}%)")
print(f"  {'─'*50}")
print(f"  Total:                  {total_time*1000:>8.1f}ms")

print("\n💰 Cost Breakdown (per query):\n")
print(f"  Query Embedding (Titan):        ${0.00002:.6f}")
print(f"  Reranking ({RETRIEVAL_TOP_N} × Haiku):        ${RETRIEVAL_TOP_N * 0.00025:.5f}")
print(f"  Answer Generation (Opus):       ${0.05:.5f}")
print(f"  {'─'*50}")
total_cost = 0.00002 + (RETRIEVAL_TOP_N * 0.00025) + 0.05
print(f"  Total per query:                ${total_cost:.5f}")

print("\n📊 Comparison with Simple RAG:\n")
simple_cost = 0.00002 + 0.05
print(f"  Simple RAG cost:                ${simple_cost:.5f}")
print(f"  Reranking cost:                 ${total_cost:.5f}")
print(f"  Additional cost:                ${total_cost - simple_cost:.5f} ({(total_cost/simple_cost - 1)*100:.1f}% increase)")
print(f"  Additional latency:             ~{rerank_time:.1f}s")

print("\n🎯 Trade-off Analysis:")
print(f"  Cost increase: {(total_cost/simple_cost - 1)*100:.0f}%")
print(f"  Latency increase: ~{rerank_time/total_time*100:.0f}%")
print(f"  Expected precision improvement: 10-30%")
print(f"  Best for: Production systems prioritizing accuracy")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Two-stage retrieval pipeline
✅ LLM-based relevance scoring
✅ Fast candidate retrieval
✅ Precision-focused reranking
✅ Performance and cost analysis

### Performance Characteristics

- **Latency**: +1-3 seconds (reranking overhead)
- **Cost**: +$0.005 per query (20 Haiku calls)
- **Precision**: +10-30% improvement
- **Recall**: Same as initial retrieval

### When to Use Reranking

**Use Reranking when:**
- Precision is critical
- Vector search returns false positives
- Queries have subtle intent
- Cost increase is acceptable
- Production RAG systems

**Skip Reranking when:**
- Latency must be minimal
- Budget is very tight
- Vector search performs well
- Simple factual lookups
- Prototyping/development

### Reranking Strategies

**1. LLM-based (This Notebook)**
- Most accurate
- Understands nuance
- Higher cost
- Slower

**2. Cross-encoder Models**
- Specialized rerankers
- Fast and cheap
- Less flexible
- Requires deployment

**3. Hybrid Scoring**
- Combine vector + keyword + metadata
- No LLM needed
- Fast
- Less accurate

### Optimization Tips

1. **Use Haiku for scoring**: 20x cheaper than Opus
2. **Batch reranking**: Score multiple docs per call
3. **Cache scores**: For similar queries
4. **Adaptive reranking**: Only for complex queries
5. **Parallel scoring**: Score docs concurrently

### Best Practices

1. **Cast wide net initially**: Retrieve 2-4x final top-k
2. **Clear scoring criteria**: Explicit relevance rubric
3. **Temperature = 0**: Consistent scoring
4. **Monitor costs**: Track per-query expenses
5. **A/B test**: Measure actual improvement

### Limitations

1. **Higher latency**: 1-3s overhead
2. **Higher cost**: Proportional to candidate count
3. **LLM biases**: Scoring may be inconsistent
4. **Diminishing returns**: Beyond 20-30 candidates

### Advanced Techniques

- **Listwise reranking**: Score all docs together
- **Pairwise comparison**: Compare docs pairwise
- **Multi-stage reranking**: Coarse → fine filtering
- **Ensemble reranking**: Combine multiple rerankers
- **Learned reranking**: Fine-tune reranker model

### Next Steps

- **Combine with Fusion**: Rerank fused results
- **Add metadata**: Use doc metadata in scoring
- **Cross-encoder**: Deploy specialized reranker
- **Hybrid scoring**: Weighted combination

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")